In [1]:
import requests
import pandas as pd
import json
import os
import boto3
import scrapy
from dotenv import load_dotenv


csv_url="https://tmopenlabbucket.s3.eu-west-3.amazonaws.com/City_Meteo_Rank.csv"


df = pd.read_csv(csv_url,index_col=0)

In [24]:
list_cities=df['city'].to_list()

In [25]:
import subprocess


# Liste des villes à envoyer à script.py
cities = list_cities

# Construire l'argument en ligne de commande
cmd = ['f:\\Informatique\\Jedha\\env\\Scripts\\python.exe','booking_scrap_final.py','--cities'] + cities

# Exécuter le script en passant les villes comme argument
result = subprocess.run(cmd)


In [2]:
df_booking=pd.read_json('hotels.json')
df_city_ccm=pd.read_csv('cities_lat_long_ccm.csv',index_col=0)


In [3]:
df_city_ccm

,city,lat,lon,CCM
20,Marseille,43.296174,5.369953,0.917
18,Bormes les Mimosas,43.150697,6.341928,0.910
25,Aigues Mortes,43.566152,4.191540,0.896
24,Nimes,43.837425,4.360069,0.872
22,Avignon,43.949249,4.805901,0.870
19,Cassis,43.214036,5.539632,0.867
28,Carcassonne,43.213036,2.349107,0.856
33,La Rochelle,46.159113,-1.152043,0.849
31,Biarritz,43.471144,-1.552727,0.847
21,Aix en Provence,43.529842,5.447474,0.843


In [4]:
df_booking_grouped = df_booking.groupby('city').agg(lambda x: list(x)).reset_index()
df_joined=df_booking_grouped.join(df_city_ccm.set_index('city'), on='city')
df_joined.columns=['City','Hotels_name','Hotels_url','Hotels_score','Hotels_description','Hotels_latitude','Hotels_longitude','City_latitude','City_longitude','City_CCM']
df_joined.head()

,City,Hotels_name,Hotels_url,Hotels_score,Hotels_description,Hotels_latitude,Hotels_longitude,City_latitude,City_longitude,City_CCM
0,Aigues Mortes,"[Hôtel Le Médiéval, Maison Arthur, Au Cœur des...",[https://www.booking.com/hotel/fr/le-medieval....,"[8.7, 9.0, 9.9, 9.5, 8.5, 9.2, 8.4, 9.1, 9.4, ...",[Le Médiéval is located on the banks of the ca...,"[43.57186625, 43.5659428, 43.565401, 43.570192...","[4.19366169, 4.1924, 4.192973, 4.1950814, 4.18...",43.566152,4.191540,0.896
1,Aix en Provence,"[Aix Homes ""Les Allées Provençales"", Aix Homes...",[https://www.booking.com/hotel/fr/aix-homes-1....,"[9.2, 8.7, 8.8, 9.1, 8.1, 8.7, 9.0, 9.1, 8.2, ...","[Set in Aix-en-Provence, the recently renovate...","[43.5256056, 43.5287484, 43.5272537, 43.537268...","[5.4409509, 5.4436653, 5.4452585, 5.4486186, 5...",43.529842,5.447474,0.843
2,Amiens,[DOWNTOWN LOFT - CENTRE VILLE - WiFi - NETFLIX...,[https://www.booking.com/hotel/fr/downtown-lof...,"[9.6, 9.0, 7.7, 10.0, 9.1, 8.2, 8.9, 9.0, 8.3,...","[Situated 1.2 km from Amiens Train Station, 2....","[49.89219043, 49.8926758, 49.89125423, 49.8936...","[2.29471652, 2.3082578, 2.30433606, 2.301612, ...",49.894171,2.295695,0.679
3,Annecy,[Highly recommended apartment steps from the l...,[https://www.booking.com/hotel/fr/annecy-vacat...,"[9.2, 9.4, 7.9, 8.6, 9.0, 8.2, 8.6, 9.0, 9.3, ...",[Highly recommended apartment steps from the l...,"[45.901008, 45.90584116, 45.88969452, 45.90393...","[6.126982, 6.14062912, 6.13690495, 6.11999363,...",45.899235,6.128885,0.683
4,Avignon,"[My Pad Provence 6, Joli Studio Avec Jardinet,...",[https://www.booking.com/hotel/fr/my-pad-prove...,"[9.3, 8.5, 9.3, 9.3, 8.1, 8.0, 8.8, 8.6, 9.0, ...",[In the Avignon City Centre district of Avigno...,"[43.9515236, 43.94182031, 43.9361955, 43.95065...","[4.8170918, 4.81495336, 4.82600075, 4.8033086,...",43.949249,4.805901,0.870


In [5]:
# Réorganiser les colonnes de df_joined
columns_order = ['City', 'City_latitude', 'City_longitude', 'City_CCM', 
                 'Hotels_name', 'Hotels_url', 'Hotels_score', 'Hotels_description', 
                 'Hotels_latitude', 'Hotels_longitude']

df_joined = df_joined[columns_order]
df_joined.head()

,City,City_latitude,City_longitude,City_CCM,Hotels_name,Hotels_url,Hotels_score,Hotels_description,Hotels_latitude,Hotels_longitude
0,Aigues Mortes,43.566152,4.191540,0.896,"[Hôtel Le Médiéval, Maison Arthur, Au Cœur des...",[https://www.booking.com/hotel/fr/le-medieval....,"[8.7, 9.0, 9.9, 9.5, 8.5, 9.2, 8.4, 9.1, 9.4, ...",[Le Médiéval is located on the banks of the ca...,"[43.57186625, 43.5659428, 43.565401, 43.570192...","[4.19366169, 4.1924, 4.192973, 4.1950814, 4.18..."
1,Aix en Provence,43.529842,5.447474,0.843,"[Aix Homes ""Les Allées Provençales"", Aix Homes...",[https://www.booking.com/hotel/fr/aix-homes-1....,"[9.2, 8.7, 8.8, 9.1, 8.1, 8.7, 9.0, 9.1, 8.2, ...","[Set in Aix-en-Provence, the recently renovate...","[43.5256056, 43.5287484, 43.5272537, 43.537268...","[5.4409509, 5.4436653, 5.4452585, 5.4486186, 5..."
2,Amiens,49.894171,2.295695,0.679,[DOWNTOWN LOFT - CENTRE VILLE - WiFi - NETFLIX...,[https://www.booking.com/hotel/fr/downtown-lof...,"[9.6, 9.0, 7.7, 10.0, 9.1, 8.2, 8.9, 9.0, 8.3,...","[Situated 1.2 km from Amiens Train Station, 2....","[49.89219043, 49.8926758, 49.89125423, 49.8936...","[2.29471652, 2.3082578, 2.30433606, 2.301612, ..."
3,Annecy,45.899235,6.128885,0.683,[Highly recommended apartment steps from the l...,[https://www.booking.com/hotel/fr/annecy-vacat...,"[9.2, 9.4, 7.9, 8.6, 9.0, 8.2, 8.6, 9.0, 9.3, ...",[Highly recommended apartment steps from the l...,"[45.901008, 45.90584116, 45.88969452, 45.90393...","[6.126982, 6.14062912, 6.13690495, 6.11999363,..."
4,Avignon,43.949249,4.805901,0.870,"[My Pad Provence 6, Joli Studio Avec Jardinet,...",[https://www.booking.com/hotel/fr/my-pad-prove...,"[9.3, 8.5, 9.3, 9.3, 8.1, 8.0, 8.8, 8.6, 9.0, ...",[In the Avignon City Centre district of Avigno...,"[43.9515236, 43.94182031, 43.9361955, 43.95065...","[4.8170918, 4.81495336, 4.82600075, 4.8033086,..."


In [6]:
df_joined.to_csv('City_Meteo_Rank_Booking.csv')

In [7]:
load_dotenv()

aws_access_key_id =os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key =os.getenv('AWS_SECRET_ACCESS_KEY')
s3 = boto3.resource('s3',aws_access_key_id=aws_access_key_id,aws_secret_access_key=aws_secret_access_key)
bucket=s3.Bucket('tmopenlabbucket')

In [8]:
bucket.upload_file(Filename='City_Meteo_Rank_Booking.csv',Key='City_Meteo_Rank_Booking.csv', ExtraArgs={'ACL':'public-read'})